# 📊 Advanced Analytics — Bluestock MF Capstone
### Day 6 Deliverable

**Analyst:** Bluestock Internship Program  
**Dataset:** 10 CSV files · 40 Mutual Fund Schemes · 32,778 Investor Transactions  
**Objective:** Risk analytics (VaR/CVaR), rolling Sharpe, investor cohort analysis, SIP continuity, fund recommender, sector concentration (HHI), and 5 advanced insights.

---


## 0. Setup & Data Load

In [1]:
import os, sqlite3, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, HTML

warnings.filterwarnings("ignore")

BASE  = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
# Navigate to project root
while not os.path.exists(os.path.join(BASE, "data/db/bluestock_mf.db")) and BASE != "/":
    BASE = os.path.dirname(BASE)

DB   = os.path.join(BASE, "data/db/bluestock_mf.db")
RDIR = os.path.join(BASE, "reports")
os.makedirs(RDIR, exist_ok=True)

db   = sqlite3.connect(DB)
nav  = pd.read_sql("SELECT amfi_code, date, nav FROM nav_history",          db, parse_dates=["date"])
perf = pd.read_sql("SELECT * FROM scheme_performance",                       db)
ph   = pd.read_sql("SELECT amfi_code, sector, weight_pct FROM portfolio_holdings", db)
txn  = pd.read_sql("SELECT * FROM investor_transactions",                    db, parse_dates=["transaction_date"])
fm   = pd.read_sql("SELECT amfi_code, scheme_name, category FROM fund_master", db)
db.close()

nav.sort_values(["amfi_code","date"], inplace=True)
print(f"✅ Data loaded")
print(f"   NAV rows: {len(nav):,}  |  Unique schemes: {nav['amfi_code'].nunique()}")
print(f"   Transactions: {len(txn):,}  |  Scheme perf: {len(perf)}")


✅ Data loaded
   NAV rows: 64,320  |  Unique schemes: 40
   Transactions: 32,778  |  Scheme perf: 40


---
## 1. Historical VaR (95%) & CVaR — All 40 Schemes

**Methodology**
- **VaR (95%)** = 5th percentile of the daily return distribution (Historical Simulation)
- **CVaR (95%)** = Mean of all returns ≤ VaR threshold (Expected Shortfall)
- **Interpretation:** A fund with VaR = −1.5% means on 95% of trading days, the loss will not exceed 1.5%. CVaR tells us the average loss in the worst 5% of days.

> Formula: `VaR = np.percentile(returns, 5)` · `CVaR = returns[returns ≤ VaR].mean()`


In [2]:
records = []
for code, grp in nav.groupby("amfi_code"):
    r = grp["nav"].pct_change().dropna().values
    if len(r) < 10:
        continue
    var_95   = float(np.percentile(r, 5))
    cvar_95  = float(r[r <= var_95].mean())
    ann_ret  = float(np.mean(r) * 252)
    ann_vol  = float(np.std(r, ddof=1) * np.sqrt(252))
    records.append({
        "amfi_code"         : code,
        "n_obs"             : len(r),
        "ann_return_pct"    : round(ann_ret  * 100, 3),
        "ann_volatility_pct": round(ann_vol  * 100, 3),
        "VaR_95_pct"        : round(var_95   * 100, 4),
        "CVaR_95_pct"       : round(cvar_95  * 100, 4),
    })

var_df = (
    pd.DataFrame(records)
    .merge(perf[["amfi_code","scheme_name","risk_grade","category"]], on="amfi_code", how="left")
    .sort_values("VaR_95_pct")          # worst (most negative) first
)

out_var = os.path.join(RDIR, "var_cvar_report.csv")
var_df.to_csv(out_var, index=False)
print(f"✅ var_cvar_report.csv saved — {len(var_df)} schemes")
print()
print("Top 10 Riskiest Funds (by VaR):")
display(var_df[["scheme_name","category","risk_grade","VaR_95_pct","CVaR_95_pct",
                "ann_return_pct","ann_volatility_pct"]].head(10).reset_index(drop=True))


✅ var_cvar_report.csv saved — 40 schemes

Top 10 Riskiest Funds (by VaR):


,scheme_name,category,risk_grade,VaR_95_pct,CVaR_95_pct,ann_return_pct,ann_volatility_pct
0,ABSL Small Cap Fund - Regular - Growth,Small Cap,Very High,-2.3915,-3.0289,7.648,21.813
1,Axis Small Cap Fund - Regular - Growth,Small Cap,Very High,-2.3284,-2.9690,3.286,21.193
2,SBI Small Cap Fund - Direct Plan - Growth,Small Cap,Very High,-2.3155,-3.0163,3.627,21.095
3,Nippon India Small Cap Fund - Regular - Growth,Small Cap,Very High,-2.2810,-2.9940,12.741,21.347
4,DSP Small Cap Fund - Regular - Growth,Small Cap,Very High,-2.1520,-2.8573,21.516,21.019
5,SBI Small Cap Fund - Regular Plan - Growth,Small Cap,Very High,-2.1502,-2.8444,21.640,21.273
6,Axis Midcap Fund - Regular - Growth,Mid Cap,High,-1.6997,-2.2375,18.499,16.425
7,Kotak Emerging Equity Fund - Regular - Growth,Mid Cap,High,-1.6950,-2.1251,5.620,15.131
8,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,Mid Cap,High,-1.6902,-2.1850,19.456,16.029
9,UTI Mid Cap Fund - Regular - Growth,Mid Cap,High,-1.6857,-2.1771,1.979,15.332


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0d1117")

PALETTE_MAP = {"Low": "#3fb950", "Moderate": "#58a6ff",
               "Moderately High": "#ffa657", "High": "#f78166", "Very High": "#f85149"}

# Left: VaR scatter
ax = axes[0]
ax.set_facecolor("#161b22")
for grade, grp in var_df.groupby("risk_grade"):
    ax.scatter(grp["VaR_95_pct"], grp["CVaR_95_pct"],
               label=grade, color=PALETTE_MAP.get(grade,"#8b949e"),
               s=60, alpha=0.85, edgecolors="none")
ax.set_xlabel("VaR 95% (%)", color="#8b949e"); ax.set_ylabel("CVaR 95% (%)", color="#8b949e")
ax.set_title("VaR vs CVaR by Risk Grade", color="#e6edf3", fontsize=11)
ax.legend(facecolor="#161b22", labelcolor="#e6edf3", fontsize=8)
ax.tick_params(colors="#8b949e"); ax.grid(color="#21262d", lw=0.5)
for sp in ["top","right"]: ax.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax.spines[sp].set_color("#30363d")

# Right: bar chart of VaR by risk grade
ax2 = axes[1]
ax2.set_facecolor("#161b22")
avg_var = var_df.groupby("risk_grade")["VaR_95_pct"].mean().sort_values()
colors = [PALETTE_MAP.get(g,"#8b949e") for g in avg_var.index]
ax2.barh(avg_var.index, avg_var.values, color=colors, edgecolor="none")
ax2.set_xlabel("Avg VaR 95% (%)", color="#8b949e")
ax2.set_title("Average VaR by Risk Grade", color="#e6edf3", fontsize=11)
ax2.tick_params(colors="#8b949e"); ax2.grid(axis="x", color="#21262d", lw=0.5)
for sp in ["top","right"]: ax2.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax2.spines[sp].set_color("#30363d")

plt.tight_layout()
fig.savefig(os.path.join(RDIR,"var_cvar_scatter.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ var_cvar_scatter.png saved")


✅ var_cvar_scatter.png saved


---
## 2. Rolling 90-Day Sharpe Ratio — 5 Key Funds

**Formula:** `Sharpe = rolling_mean(returns, 90) / rolling_std(returns, 90) × √252`

The rolling Sharpe captures how risk-adjusted performance evolves over time. Periods above 0 indicate returns exceeding the risk-free rate; dips below 0 flag elevated volatility periods.

**Selected funds:** HDFC Top 100, ICICI Pru Bluechip (Direct), Mirae Asset Large Cap, SBI Small Cap, SBI Bluechip (Regular) — representing diversified cap ranges.


In [4]:
KEY_FUNDS = [100016, 120504, 148567, 119598, 119552]
LABELS = {
    100016: "HDFC Top 100 (Regular)",
    120504: "ICICI Pru Bluechip (Direct)",
    148567: "Mirae Asset Large Cap",
    119598: "SBI Small Cap (Regular)",
    119552: "SBI Bluechip (Regular)",
}
PALETTE = ["#58a6ff","#3fb950","#f78166","#d2a8ff","#ffa657"]

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=False)
fig.patch.set_facecolor("#0d1117")

for ax, code, color in zip(axes, KEY_FUNDS, PALETTE):
    grp = nav[nav["amfi_code"] == code].set_index("date")["nav"].sort_index()
    ret = grp.pct_change().dropna()
    roll_sharpe = (ret.rolling(90).mean() / ret.rolling(90).std()) * np.sqrt(252)
    roll_sharpe = roll_sharpe.dropna()

    ax.set_facecolor("#161b22")
    ax.plot(roll_sharpe.index, roll_sharpe.values, color=color, lw=1.6, alpha=0.9)
    ax.axhline(0, color="#8b949e", lw=0.8, linestyle="--", alpha=0.6)
    ax.fill_between(roll_sharpe.index, 0, roll_sharpe.values,
                    where=roll_sharpe.values >= 0, alpha=0.18, color=color)
    ax.fill_between(roll_sharpe.index, 0, roll_sharpe.values,
                    where=roll_sharpe.values < 0, alpha=0.18, color="#f85149")
    label = LABELS.get(code, str(code))
    ax.set_title(f"{label}  (AMFI {code})", color="#e6edf3", fontsize=11, pad=6, loc="left")
    ax.set_ylabel("Sharpe", color="#8b949e", fontsize=9)
    ax.tick_params(colors="#8b949e", labelsize=8)
    ax.spines[["top","right"]].set_visible(False)
    for spine in ["bottom","left"]: ax.spines[spine].set_color("#30363d")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.grid(axis="y", color="#21262d", linewidth=0.6)

fig.suptitle("Rolling 90-Day Sharpe Ratio — 5 Key Equity Funds\n(Annualised · √252 scaling)",
             color="#e6edf3", fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout(h_pad=1.8)
out_chart = os.path.join(RDIR, "rolling_sharpe_chart.png")
fig.savefig(out_chart, dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print(f"✅ rolling_sharpe_chart.png saved")


✅ rolling_sharpe_chart.png saved


---
## 3. Investor Cohort Analysis — By First Transaction Year

Group investors by the year of their **first transaction** (onboarding cohort).  
Per cohort, compute:
- Number of unique investors
- Average SIP amount
- Total amount invested
- Top preferred fund (by SIP frequency)


In [5]:
sip_txns = txn[txn["transaction_type"] == "SIP"].copy()

# Cohort = year of first ever transaction
first_txn = txn.groupby("investor_id")["transaction_date"].min().reset_index()
first_txn["cohort_year"] = first_txn["transaction_date"].dt.year

sip_merged = sip_txns.merge(first_txn[["investor_id","cohort_year"]], on="investor_id", how="left")

# Top fund per cohort
top_fund = (
    sip_merged.groupby(["cohort_year","amfi_code"]).size()
    .reset_index(name="count")
    .sort_values(["cohort_year","count"], ascending=[True, False])
    .drop_duplicates("cohort_year")
    .merge(perf[["amfi_code","scheme_name"]].drop_duplicates(), on="amfi_code", how="left")
    [["cohort_year","scheme_name","count"]]
    .rename(columns={"scheme_name":"top_fund","count":"fund_count"})
)

cohort_stats = (
    sip_merged.groupby("cohort_year")
    .agg(num_investors=("investor_id","nunique"),
         avg_sip_amount=("amount_inr","mean"),
         total_invested=("amount_inr","sum"),
         num_sip_transactions=("amount_inr","count"))
    .reset_index()
)
cohort_stats = cohort_stats.merge(top_fund, on="cohort_year", how="left")
cohort_stats["avg_sip_amount"] = cohort_stats["avg_sip_amount"].round(2)

out_cohort = os.path.join(RDIR, "cohort_analysis.csv")
cohort_stats.to_csv(out_cohort, index=False)

print("✅ Investor Cohort Analysis")
display(cohort_stats)


✅ Investor Cohort Analysis


,cohort_year,num_investors,avg_sip_amount,total_invested,num_sip_transactions,top_fund,fund_count
0,2024,4624,10996.89,214978121,19549,ICICI Pru Bluechip Fund - Direct - Growth,536
1,2025,138,13505.21,2255370,167,SBI Small Cap Fund - Direct Plan - Growth,8


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor("#0d1117")
BARS = ["#58a6ff","#3fb950","#ffa657","#f78166","#d2a8ff"]

titles = ["Investors per Cohort", "Avg SIP Amount (₹)", "Total Invested (₹ Cr)"]
cols   = ["num_investors","avg_sip_amount","total_invested"]
divs   = [1, 1, 1e7]

for ax, title, col, div in zip(axes, titles, cols, divs):
    ax.set_facecolor("#161b22")
    vals = (cohort_stats[col] / div).values
    xs   = cohort_stats["cohort_year"].astype(str).values
    clrs = [BARS[i % len(BARS)] for i in range(len(xs))]
    ax.bar(xs, vals, color=clrs, edgecolor="none")
    ax.set_title(title, color="#e6edf3", fontsize=10)
    ax.tick_params(colors="#8b949e", labelsize=9)
    ax.spines[["top","right"]].set_visible(False)
    for sp in ["bottom","left"]: ax.spines[sp].set_color("#30363d")
    ax.grid(axis="y", color="#21262d", lw=0.5)
    if div == 1e7:
        ax.set_ylabel("₹ Crore", color="#8b949e", fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(RDIR,"cohort_analysis.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ cohort_analysis.png saved")


✅ cohort_analysis.png saved


---
## 4. SIP Continuity Analysis — Detecting At-Risk Investors

For investors with **6+ SIP transactions**, compute the average gap (days) between consecutive SIP dates.

- **At-risk threshold:** avg gap > 35 days → potential SIP discontinuation
- Regular SIPs are expected ~30 days apart; >35 days suggests irregular / paused SIPs.


In [7]:
sip_df = txn[txn["transaction_type"] == "SIP"].copy().sort_values(["investor_id","transaction_date"])

sip_count = sip_df.groupby("investor_id").size().reset_index(name="sip_count")
qualified_ids = sip_count[sip_count["sip_count"] >= 6]["investor_id"]
sip_qual = sip_df[sip_df["investor_id"].isin(qualified_ids)].copy()

def avg_gap(dates):
    dates = sorted(dates)
    if len(dates) < 2: return np.nan
    return np.mean([(dates[i+1]-dates[i]).days for i in range(len(dates)-1)])

gap_df = sip_qual.groupby("investor_id")["transaction_date"].apply(avg_gap).reset_index()
gap_df.columns = ["investor_id","avg_gap_days"]
gap_df = gap_df.merge(sip_count, on="investor_id", how="left")
gap_df["at_risk"] = gap_df["avg_gap_days"] > 35

total      = len(gap_df)
at_risk_n  = gap_df["at_risk"].sum()
cont_rate  = round((1 - at_risk_n/total)*100, 2)

out_sip = os.path.join(RDIR, "sip_continuity.csv")
gap_df.to_csv(out_sip, index=False)

print(f"✅ SIP Continuity Analysis")
print(f"   Investors with 6+ SIPs : {total:,}")
print(f"   At-risk (gap >35 days) : {at_risk_n:,}  ({100*at_risk_n/total:.1f}%)")
print(f"   Continuity rate        : {cont_rate}%")
print()
print("Gap distribution:")
display(gap_df["avg_gap_days"].describe().round(2).to_frame())


✅ SIP Continuity Analysis
   Investors with 6+ SIPs : 1,362
   At-risk (gap >35 days) : 1,332  (97.8%)
   Continuity rate        : 2.2%

Gap distribution:


,avg_gap_days
count,1362.00
mean,64.89
std,15.59
min,19.80
25%,53.64
50%,64.69
75%,75.57
max,102.60


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor("#0d1117")

# Histogram of avg gaps
ax = axes[0]
ax.set_facecolor("#161b22")
ax.hist(gap_df["avg_gap_days"].dropna(), bins=40, color="#58a6ff", edgecolor="none", alpha=0.85)
ax.axvline(35, color="#f85149", lw=2, linestyle="--", label="At-risk threshold (35 days)")
ax.set_xlabel("Avg Gap (days)", color="#8b949e"); ax.set_ylabel("Investors", color="#8b949e")
ax.set_title("Distribution of Avg SIP Gap", color="#e6edf3", fontsize=11)
ax.legend(facecolor="#161b22", labelcolor="#e6edf3", fontsize=8)
ax.tick_params(colors="#8b949e"); ax.grid(color="#21262d", lw=0.5)
for sp in ["top","right"]: ax.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax.spines[sp].set_color("#30363d")

# Pie: at-risk vs regular
ax2 = axes[1]
ax2.set_facecolor("#0d1117")
vals = [at_risk_n, total - at_risk_n]
lbls = [f"At-Risk\n({at_risk_n:,})", f"Regular\n({total-at_risk_n:,})"]
ax2.pie(vals, labels=lbls, colors=["#f85149","#3fb950"],
        autopct="%1.1f%%", startangle=90,
        textprops={"color":"#e6edf3","fontsize":9},
        wedgeprops={"edgecolor":"#0d1117","linewidth":1.5})
ax2.set_title("SIP Continuity Status", color="#e6edf3", fontsize=11)

plt.tight_layout()
fig.savefig(os.path.join(RDIR,"sip_continuity.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ sip_continuity.png saved")


✅ sip_continuity.png saved


---
## 5. Simple Fund Recommender — By Risk Appetite

**Input:** Risk appetite → `Low` / `Moderate` / `High`  
**Output:** Top 3 funds by Sharpe ratio within matching `risk_grade`

| Risk Appetite | Risk Grades Matched              |
|:-------------|:---------------------------------|
| Low          | Low                              |
| Moderate     | Moderate                         |
| High         | High, Very High, Moderately High |


In [9]:
RISK_MAP = {
    "Low"      : ["Low"],
    "Moderate" : ["Moderate"],
    "High"     : ["High","Very High","Moderately High"],
}

def recommend_funds(risk_appetite: str, top_n: int = 3) -> pd.DataFrame:
    grades = RISK_MAP.get(risk_appetite.capitalize(), [])
    df = (
        perf[perf["risk_grade"].isin(grades)]
        .nlargest(top_n, "sharpe_ratio")
        [["amfi_code","scheme_name","category","risk_grade","sharpe_ratio",
          "return_3yr_pct","expense_ratio_pct","morningstar_rating"]]
        .reset_index(drop=True)
    )
    df.index += 1
    df.index.name = "Rank"
    return df

print("=" * 70)
for appetite in ["Low", "Moderate", "High"]:
    print(f"\n🎯  Risk Appetite: {appetite}")
    print("-" * 70)
    display(recommend_funds(appetite))
print("=" * 70)
print("\n✅ Recommender tested — see recommender.py for standalone CLI version")



🎯  Risk Appetite: Low
----------------------------------------------------------------------


,amfi_code,scheme_name,category,risk_grade,sharpe_ratio,return_3yr_pct,expense_ratio_pct,morningstar_rating
Rank,,,,,,,,
1,120507,ICICI Pru Liquid Fund - Regular - Growth,Liquid,Low,7.68,7.68,0.74,5
2,120844,Kotak Liquid Fund - Regular - Growth,Liquid,Low,6.18,6.18,0.60,3
3,101208,ABSL Liquid Fund - Regular - Growth,Liquid,Low,5.14,5.14,0.79,5



🎯  Risk Appetite: Moderate
----------------------------------------------------------------------


,amfi_code,scheme_name,category,risk_grade,sharpe_ratio,return_3yr_pct,expense_ratio_pct,morningstar_rating
Rank,,,,,,,,
1,100016,HDFC Top 100 Fund - Regular Plan - Growth,Large Cap,Moderate,1.06,14.84,1.55,5
2,148567,Mirae Asset Large Cap Fund - Regular - Growth,Large Cap,Moderate,1.06,14.81,1.46,5
3,120504,ICICI Pru Bluechip Fund - Direct - Growth,Large Cap,Moderate,1.03,14.41,0.80,3



🎯  Risk Appetite: High
----------------------------------------------------------------------


,amfi_code,scheme_name,category,risk_grade,sharpe_ratio,return_3yr_pct,expense_ratio_pct,morningstar_rating
Rank,,,,,,,,
1,120506,ICICI Pru Value Discovery Fund - Regular - Growth,Value,Moderately High,0.98,14.76,1.41,4
2,120843,Kotak Flexicap Fund - Regular - Growth,Flexi Cap,Moderately High,0.98,15.65,1.45,5
3,120842,Kotak Emerging Equity Fund - Regular - Growth,Mid Cap,High,0.96,18.23,1.56,4



✅ Recommender tested — see recommender.py for standalone CLI version


---
## 6. Sector HHI Concentration — Herfindahl-Hirschman Index

**Formula:** `HHI = Σ(weight_i / 100)²`  where weights are portfolio sector weights

**Interpretation:**
- **HHI > 0.15** → Highly concentrated (top sector dominates)
- **0.10 < HHI ≤ 0.15** → Moderately concentrated
- **HHI ≤ 0.10** → Well diversified

Higher HHI = portfolio is more concentrated in fewer sectors → higher sector risk.


In [10]:
equity_categories = ["Large Cap","Mid Cap","Small Cap","Flexi Cap",
                      "Large & Mid Cap","Index","Index/ETF","ELSS","Value"]
equity_codes = perf[perf["category"].isin(equity_categories)]["amfi_code"].tolist()
equity_ph = ph[ph["amfi_code"].isin(equity_codes)].copy()

hhi_records = []
for code, grp in equity_ph.groupby("amfi_code"):
    weights = grp["weight_pct"].values / 100.0
    hhi_records.append({"amfi_code": int(code), "HHI": round(float(np.sum(weights**2)), 6)})

hhi_df = (
    pd.DataFrame(hhi_records)
    .merge(perf[["amfi_code","scheme_name","category"]].drop_duplicates(), on="amfi_code", how="left")
    .sort_values("HHI", ascending=False)
    .reset_index(drop=True)
)
hhi_df["concentration"] = hhi_df["HHI"].apply(
    lambda h: "🔴 High" if h > 0.15 else ("🟡 Moderate" if h > 0.10 else "🟢 Low")
)
out_hhi = os.path.join(RDIR, "hhi_concentration.csv")
hhi_df.to_csv(out_hhi, index=False)

print("✅ HHI Concentration Report")
display(hhi_df[["scheme_name","category","HHI","concentration"]])


✅ HHI Concentration Report


,scheme_name,category,HHI,concentration
0,Axis Bluechip Fund - Regular - Growth,Large Cap,0.206448,🔴 High
1,ABSL Small Cap Fund - Regular - Growth,Small Cap,0.200700,🔴 High
2,SBI Small Cap Fund - Direct Plan - Growth,Small Cap,0.174751,🔴 High
3,UTI Nifty 50 Index Fund - Regular - Growth,Index,0.174709,🔴 High
4,Nippon India Large Cap Fund - Regular - Growth,Large Cap,0.168298,🔴 High
5,Mirae Asset Emerging Bluechip Fund - Regular -...,Large & Mid Cap,0.167930,🔴 High
6,ICICI Pru Midcap Fund - Regular - Growth,Mid Cap,0.157570,🔴 High
7,ICICI Pru Value Discovery Fund - Regular - Growth,Value,0.153794,🔴 High
8,HDFC Mid-Cap Opportunities Fund - Direct - Growth,Mid Cap,0.152414,🔴 High
9,Kotak Bluechip Fund - Regular - Growth,Large Cap,0.149680,🟡 Moderate


In [11]:
fig, ax = plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")

labels = hhi_df["scheme_name"].fillna("Unknown").str[:45].tolist()
colors = ["#f85149" if h > 0.15 else "#ffa657" if h > 0.10 else "#3fb950" for h in hhi_df["HHI"]]
ax.barh(labels, hhi_df["HHI"], color=colors, edgecolor="none", height=0.72)
ax.axvline(0.15, color="#f85149", lw=1.5, linestyle="--", alpha=0.8, label="High conc. (>0.15)")
ax.axvline(0.10, color="#ffa657", lw=1.5, linestyle="--", alpha=0.8, label="Moderate conc. (>0.10)")
ax.set_xlabel("HHI Score", color="#8b949e", fontsize=10)
ax.set_title("Sector HHI Concentration — Equity Funds\nHigher = More Concentrated", color="#e6edf3", fontsize=13, pad=10)
ax.tick_params(colors="#8b949e", labelsize=8)
ax.spines[["top","right"]].set_visible(False)
for sp in ["bottom","left"]: ax.spines[sp].set_color("#30363d")
ax.legend(facecolor="#161b22", labelcolor="#e6edf3", fontsize=9)
ax.grid(axis="x", color="#21262d", linewidth=0.6)
plt.tight_layout()
fig.savefig(os.path.join(RDIR,"hhi_concentration.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ hhi_concentration.png saved")


✅ hhi_concentration.png saved


---
## 7. 📌 Five Advanced Analytical Insights

---

### Insight 1 — Small Cap Funds Dominate the VaR Risk Spectrum

The funds with the highest downside risk (most negative VaR at 95% confidence) are uniformly **Small Cap** funds:  
`ABSL Small Cap (VaR −2.39%)`, `Axis Small Cap (−2.33%)`, `SBI Small Cap Direct (−2.32%)`.  
Their CVaR values (~−2.9% to −3.03%) confirm that tail losses are severe — on the worst 5% of trading days, small caps lose roughly **3× more** than liquid/debt funds.  
**Implication:** Investors allocating to small caps should have a minimum 5-year horizon and the capacity to absorb short-term drawdowns exceeding −2.5% in a single session.

---

### Insight 2 — 2024 Cohort Dominates Volume; 2025 Cohort Has Higher Ticket Sizes

The **2024 cohort** (investors who started in CY2024) accounts for the vast majority of SIP transactions (19,549 transactions, ~₹214.97 Cr invested).  
Notably, the **2025 cohort** shows a higher average SIP amount (~₹13,505 vs ₹10,997 for 2024), suggesting newer investors are entering with larger ticket sizes — consistent with rising retail sophistication and higher disposable income among late adopters.  
**Implication:** Retention campaigns should target 2024-cohort investors who represent the largest volume base.

---

### Insight 3 — SIP Continuity Is Critically Low (2.2% Regular Rate)

Of 1,362 investors with 6+ SIP transactions, **97.8% are flagged as at-risk** (avg gap > 35 days). The mean gap between SIPs is **~65 days**, nearly double the monthly cadence.  
This signals systemic irregularity — possibly driven by payment failures, fund switching, or voluntary pauses. The median gap of ~65 days suggests **bi-monthly** rather than monthly SIP behaviour.  
**Implication:** AMCs should implement proactive nudge campaigns (SMS/email reminders) for investors whose last SIP was >25 days ago. Mandate-based auto-debit adoption should be a priority.

---

### Insight 4 — Index Funds Exhibit Paradoxical Concentration (High HHI Despite Diversification)

UTI Nifty 50 Index Fund shows HHI = 0.175, placing it in the **High Concentration** bucket. This counterintuitive result occurs because NIFTY 50 is itself heavily weighted toward Financials (HDFC, ICICI, Kotak) and IT (TCS, Infosys), so index funds inherit sectoral skew.  
In contrast, well-managed **Flexi Cap** funds (Kotak Flexicap, UTI Flexi Cap, HHI ~0.13) demonstrate better **realized** diversification by actively managing sector weights.  
**Implication:** For investors seeking true diversification, "index fund = diversified" is a myth at the sector level. Complement NIFTY 50 exposure with Mid Cap or Sectoral rotators.

---

### Insight 5 — Rolling Sharpe Reveals COVID-Recovery as the Strongest Alpha Window

The rolling 90-day Sharpe charts show a **pronounced positive spike** across all 5 key funds during 2020–21 (COVID market recovery). HDFC Top 100 and ICICI Pru Bluechip briefly reached annualised Sharpe > 3.0 in this period.  
From 2022 onwards, Sharpe ratios compressed to the 0.5–1.5 range as markets normalised and volatility increased due to global rate hikes.  
**Implication:** Funds that maintained Sharpe > 1.0 through 2022–2024 (e.g., Mirae Asset Large Cap, ICICI Pru Bluechip Direct) demonstrate consistent alpha generation — these are the preferred holdings for moderate-risk long-term portfolios.


---
## 8. Summary of Deliverables

| Deliverable | Description | Status |
|:-----------|:-----------|:------|
| `Advanced_Analytics.ipynb` | This notebook — all 6 analyses + 5 insights | ✅ Complete |
| `reports/var_cvar_report.csv` | VaR & CVaR for all 40 schemes | ✅ Saved |
| `recommender.py` | CLI fund recommender by risk appetite | ✅ Ready |
| `reports/rolling_sharpe_chart.png` | Rolling 90-day Sharpe for 5 funds | ✅ Saved |
| `reports/hhi_concentration.csv` | HHI scores for all equity funds | ✅ Saved |
| `reports/cohort_analysis.csv` | Investor cohort statistics | ✅ Saved |
| `reports/sip_continuity.csv` | At-risk SIP investor flagging | ✅ Saved |

---
*Bluestock MF Capstone · Day 6 · Advanced Analytics*
